In [1]:
from paddleocr import PaddleOCR

c:\python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.16) or chardet (7.6.0)/charset_normalizer (3.1.0) doesn't match a supported version!
  warnings.warn(


In [2]:
ocr = PaddleOCR(
    lang='en', 
    use_textline_orientation=True, 
    enable_mkldnn=False
)

result = ocr.predict("Alice.jpg")

c:\python310\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\steph\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\steph\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\steph\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)

In [3]:
for res in result:
    res.print()

{'res': {'input_path': 'Alice.jpg', 'page_index': None, 'model_settings': {'use_doc_preprocessor': True, 'use_textline_orientation': True}, 'doc_preprocessor_res': {'input_path': None, 'page_index': None, 'model_settings': {'use_doc_orientation_classify': True, 'use_doc_unwarping': True}, 'angle': 0}, 'dt_polys': array([[[106,   0],
        ...,
        [106,  18]],

       ...,

       [[  0, 404],
        ...,
        [  0, 420]]], shape=(26, 4, 2), dtype=int16), 'text_det_params': {'limit_side_len': 64, 'limit_type': 'min', 'thresh': 0.3, 'max_side_limit': 4000, 'box_thresh': 0.6, 'unclip_ratio': 1.5}, 'text_type': 'general', 'textline_orientation_angles': array([0, ..., 0], shape=(26,)), 'text_rec_score_thresh': 0.0, 'return_word_box': False, 'rec_texts': ['CHAPTERI.', 'Down the Rabbit-Hole', 'Allice was beginning to get very tired of sitting by her sister on the bank.', 'and of having nothing to dox once or twice she had peeped into the book her', "sister was reading, but it had n

In [4]:
print(res["rec_texts"])

['CHAPTERI.', 'Down the Rabbit-Hole', 'Allice was beginning to get very tired of sitting by her sister on the bank.', 'and of having nothing to dox once or twice she had peeped into the book her', "sister was reading, but it had no pictures or conversations in it, 'and what is", "the use of a book,' thought Alice without pictures or conversation?", 'So she was considering in her own mind (as well as she could, for the hot', 'day made her feel very sleepy and stupid), whether the pleasure of', 'making a daisy-chain would be worth the trouble of getting up and picking', 'the daisies, when suddenly a White Rabbit with pink eyes ran close by her.', 'There was nothing so VERY remarkable in that; nor did Alice think it so', 'VERY much out of the way to hear the Rabbit say to itself, Oh dearl Oh', 'dear!I shall be late! (when she thought it over afterwards, it occurred to', 'her that she ought to have wondered at this, but at the time it all seemed', 'quite natural); but when the Rabbit actua

In [5]:
from ollama import generate
# Regular response
response = generate('llama3.2:3b', 'Going to upload some text')
print(response['response'])

I'm ready when you are. Go ahead and upload the text.


In [40]:
from ollama import chat

def LLM_Review (raw_ocr_text):
    structured_data = chat(
        model="llama3.2:3b",  # Must match the exact model name downloaded in Ollama
        messages=[
            {
                "role": "system", 
                "content": (
                    "You are an LLM Supervisor reviewing raw, messy OCR text. Your job is to:\n"
                    "1. Fix obvious OCR typographical errors (e.g., 'To1al' -> 'Total', '1an' -> 'Tan').\n"
                    "2. Reconstruct layout relationships that got broken by line-by-line scanning.\n"
                    "3. Ensure mathematical consistency across subtotal, tax, and totals."
                )
            },
            {"role": "user", "content": f"Here is the raw OCR output:\n\n{raw_ocr_text}"}
        ]
    )
    return structured_data

In [41]:
text = result = ocr.predict("output/HorizontalCrops/crop_im_1.png")

In [42]:
OCR_results = text[0]["rec_texts"]
print(OCR_results)


['Alice was beginning to get very tired of sitting by her sister on the bank.', 'and of having nothing to dor once or twice she had peeped into the book he', 'sister was reading, but it had no pictures or conversations in it, and what is', 'the use of a book.\' thought Alice "without pictures or conversation?', 'so she was considering in her own mind (as well as she could, for the hot', 'ay made her feel very sleepy and stupid), whether the pleasure of', 'making a daisy-chain would be worth the trouble of getting up and pickin', 'he daisies, when suddenly a White Rabbit with pink eves ran close by her']


In [53]:
# combine OCR outputs into a single string and pass into LLM
texts = OCR_results
s=" "
texts = s.join(texts)
LLM_outputs =[]
LLM_output = LLM_Review(texts)
print(texts)
print(LLM_output)
print("**********")
LLM_outputs.append(LLM_output["message"].content.split("\n")[2])

Alice was beginning to get very tired of sitting by her sister on the bank. and of having nothing to dor once or twice she had peeped into the book he sister was reading, but it had no pictures or conversations in it, and what is the use of a book.' thought Alice "without pictures or conversation? so she was considering in her own mind (as well as she could, for the hot ay made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and pickin he daisies, when suddenly a White Rabbit with pink eves ran close by her
model='llama3.2:3b' created_at='2026-09-06T21:39:09.6098172Z' done=True done_reason='stop' total_duration=4682854800 load_duration=2471300 prompt_eval_count=237 prompt_eval_duration=190136000 eval_count=371 eval_duration=4486920000 message=Message(role='assistant', content='Here is the cleaned-up text:\n\nAlice was beginning to get very tired of sitting by her sister on the bank. And of having nothing to do once

In [ ]:
# run the LLM for each OCR list item

texts = OCR_results
LLM_outputs =[]

for sentence in texts:
    LLM_output = LLM_Review(sentence)
    print(sentence)
    print(LLM_output)
    print("**********")
    LLM_outputs.append(LLM_output["message"].content.split("\n")[2])

# we need to process the llm output to get only the cleaned text

content='Here is the cleaned-up text with the obvious OCR errors corrected:\n\nThe daisies, when suddenly a white rabbit with pink eyes ran close by her.\n\nI fixed the following errors:\n\n* "White" (likely meant to be all capital "WHITE")\n* "eve" -> "eyes" (correct spelling of the word)\n* "ran" (correct spelling of the verb)\n\nThe text still looks a bit rough, as OCR can struggle with certain types of formatting and layout. I\'ll make sure to review it further to see if there\'s any way to improve the layout or readability.'

In [44]:
traa = LLM_output["message"].content

In [45]:
print(traa.split("\n")[2])

The daisies, when suddenly a White Rabbit with pink eyes ran close by her


In [55]:
for i in LLM_outputs:
    print(i)

Alice was beginning to get very tired of sitting by her sister on the bank. And of having nothing to do once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, and what is the use of a book? Thought Alice. "Without pictures or conversation?" So she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her.


In [47]:
for i in OCR_results:
    print(i)

Alice was beginning to get very tired of sitting by her sister on the bank.
and of having nothing to dor once or twice she had peeped into the book he
sister was reading, but it had no pictures or conversations in it, and what is
the use of a book.' thought Alice "without pictures or conversation?
so she was considering in her own mind (as well as she could, for the hot
ay made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and pickin
he daisies, when suddenly a White Rabbit with pink eves ran close by her


In [54]:
texts

'Alice was beginning to get very tired of sitting by her sister on the bank. and of having nothing to dor once or twice she had peeped into the book he sister was reading, but it had no pictures or conversations in it, and what is the use of a book.\' thought Alice "without pictures or conversation? so she was considering in her own mind (as well as she could, for the hot ay made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and pickin he daisies, when suddenly a White Rabbit with pink eves ran close by her'

a problem we have is that the OCR outputs a new return for each row on the page as a new list item but each row can be in text a continuation of a sentence from the previous row.

I need to clean this notebook up